# 04 · Regularization: Making the Inverse Problem Well-Posed

Notebook 03 ended with a disaster: unregularized least squares fit the noise and
produced wildly oscillating slip. The fix is to tell the inversion what a
*reasonable* model looks like — to add **prior information**. That is
regularization.

## Learning objectives
- Write the regularized least-squares objective.
- Understand the three regularization operators `geodef` offers: **smoothing**
  (Laplacian), **damping** (minimum norm), and the physically-motivated
  **stress kernel**.
- Control the strength with `smoothing_strength` ($\lambda$) and see the effect
  of under- vs. over-smoothing.
- Regularize toward a nonzero reference model with `smoothing_target`.

## Tikhonov regularization

We add a penalty term to the misfit that punishes "bad" models:

$$ \min_{\mathbf{m}} \;
\underbrace{\lVert G\mathbf{m} - \mathbf{d}\rVert^2_{C_d}}_{\text{fit the data}}
\;+\;
\underbrace{\lambda^2\,\lVert L(\mathbf{m} - \mathbf{m}_{\text{ref}})\rVert^2}_{\text{stay simple}}. $$

- $L$ is the **regularization operator**: it measures how "bad" a model is.
- $\lambda$ is the **regularization strength**: how much we trust the prior
  versus the data. $\lambda \to 0$ recovers the unregularized solution of
  notebook 03; $\lambda \to \infty$ forces $\mathbf{m} \to \mathbf{m}_{\text{ref}}$.
- $\mathbf{m}_{\text{ref}}$ is a **reference model** (default $\mathbf 0$).

### Choices of $L$
- **Smoothing** (`'laplacian'`): $L$ is the discrete Laplacian, which measures
  slip *roughness*. Penalizing it favors spatially smooth slip.
- **Damping** (`'damping'`): $L = I$, which measures slip *magnitude*. This is
  the minimum-norm / minimum-moment prior.
- **Stress kernel** (`'stresskernel'`): $L$ is built from inter-patch elastic
  stress interactions — a physically motivated smoothness.

### The augmented-system view
Regularization is just **extra equations**. Stacking

$$ \begin{bmatrix} G \\ \lambda L \end{bmatrix} \mathbf{m}
   = \begin{bmatrix} \mathbf{d} \\ \lambda L\,\mathbf{m}_{\text{ref}} \end{bmatrix} $$

and solving by ordinary least squares gives exactly the regularized solution.
The prior enters as synthetic "data" with weight $\lambda$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.greens(
    fault, geodef.GNSS(glon, glat, _zero, _zero, _zero, _one, _one, _one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.GNSS(
    glon, glat,
    ve=d_obs[0::3], vn=d_obs[1::3], vu=d_obs[2::3],
    se=np.full(n_sta, sigma), sn=np.full(n_sta, sigma), su=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

40 patches, 36 stations, 108 observations


## One smoothed inversion

Add `smoothing='laplacian'` and a strength. Compare to the true slip and to the
unregularized catastrophe from notebook 03.

In [2]:
raw = geodef.invert(fault, gnss, components="dip")                       # lambda = 0
sm = geodef.invert(fault, gnss, components="dip",
                   smoothing="laplacian", smoothing_strength=1.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
vmax = slip_true[N:].max()
lim_raw = abs(raw.slip_vector).max()
geodef.plot.slip(fault, slip_true[N:], ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="True", colorbar_label="Slip (m)")
geodef.plot.slip(fault, raw.slip_vector, ax=axes[1], cmap="RdBu_r",
                 vmin=-lim_raw, vmax=lim_raw, title="Unregularized",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, sm.slip_vector, ax=axes[2], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Laplacian, lambda=1",
                 colorbar_label="Slip (m)")
plt.tight_layout()

Smoothing recovers the bump. It does not reproduce the true slip exactly — some
detail is unresolvable from noisy surface data — but it is smooth, positive
where it should be, and physically plausible.

## The effect of $\lambda$

Sweep the strength from tiny to huge. Small $\lambda$ under-smooths (noisy);
large $\lambda$ over-smooths (washed out, and the moment is suppressed). The
"right" $\lambda$ balances the two — notebook 05 makes that choice quantitative.

The top row shows the recovered slip on a single shared diverging scale; the bottom row shows each solution's error (recovered $-$ true). Under-smoothing leaves noisy, sign-alternating residuals while over-smoothing leaves a broad, one-sided deficit.

In [3]:
lambdas = [0.01, 0.1, 1.0, 10.0, 100.0]
truth = slip_true[N:]
vmax = truth.max()
ncol = len(lambdas) + 1
fig, axes = plt.subplots(2, ncol, figsize=(2.6 * ncol, 5.6),
                         constrained_layout=True)

# Row 0: true + recovered slip, one shared symmetric RdBu_r scale.
geodef.plot.slip(fault, truth, ax=axes[0, 0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="True")
recovered = []
for ax, lam in zip(axes[0, 1:], lambdas):
    r = geodef.invert(fault, gnss, components="dip",
                      smoothing="laplacian", smoothing_strength=lam)
    recovered.append(r.slip_vector)
    geodef.plot.slip(fault, r.slip_vector, ax=ax, cmap="RdBu_r",
                     vmin=-vmax, vmax=vmax, colorbar=False,
                     title=f"lambda={lam:g}\nchi2={r.chi2:.2f}")

# Row 1: recovered - true (error), on their own shared symmetric scale.
diffs = [m - truth for m in recovered]
dmax = max(np.abs(d).max() for d in diffs)
axes[1, 0].axis("off")
for ax, lam, d in zip(axes[1, 1:], lambdas, diffs):
    geodef.plot.slip(fault, d, ax=ax, cmap="RdBu_r",
                     vmin=-dmax, vmax=dmax, colorbar=False,
                     title=f"error (lambda={lam:g})")

fig.colorbar(axes[0, 0].collections[-1], ax=axes[0, :], shrink=0.8,
             label="Slip (m)")
fig.colorbar(axes[1, 1].collections[-1], ax=axes[1, :], shrink=0.8,
             label="Recovered - true (m)")

## Damping vs. smoothing

Different priors encode different beliefs. Damping (`L = I`) pulls slip toward
*zero* everywhere (minimum norm), which tends to shrink the amplitude; smoothing
penalizes *roughness* but is happy with a large smooth bump. Here they are at a
comparable strength.

In [4]:
sm = geodef.invert(fault, gnss, components="dip",
                   smoothing="laplacian", smoothing_strength=1.0)
dm = geodef.invert(fault, gnss, components="dip",
                   smoothing="damping", smoothing_strength=1.0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, sm.slip_vector, ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Smoothing (Laplacian)",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, dm.slip_vector, ax=axes[1], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Damping (minimum norm)",
                 colorbar_label="Slip (m)")
plt.tight_layout()
print(f"smoothing moment magnitude: Mw {sm.Mw:.2f}")
print(f"damping   moment magnitude: Mw {dm.Mw:.2f}")

smoothing moment magnitude: Mw 7.08
damping   moment magnitude: Mw 7.11


## Exercises
1. **Sweep by eye.** From the $\lambda$ panel, which value looks best? Note your
   guess — notebook 05 will check it with objective criteria.
2. **Stress kernel.** Swap `smoothing='laplacian'` for `'stresskernel'` at
   `smoothing_strength=1.0`. How does the recovered slip compare?
3. **Reference model.** Build a `smoothing_target` array that is small and
   uniform, pass it to `invert(...)`, and describe how large $\lambda$ now pulls
   the solution.

## Checkpoint questions
1. In the objective, what does each of the two terms want, and how does
   $\lambda$ trade between them?
2. Why does damping tend to reduce the recovered moment more than smoothing?
3. How is regularization equivalent to adding synthetic equations to the system?

## Summary
- Regularization adds a prior penalty $\lambda^2\lVert L(\mathbf m - \mathbf m_{\text{ref}})\rVert^2$.
- `smoothing='laplacian'|'damping'|'stresskernel'` selects the operator $L$.
- $\lambda$ (`smoothing_strength`) tunes under- vs. over-smoothing.

**Next:** notebook 05 chooses $\lambda$ objectively with the L-curve, ABIC, and
cross-validation.